We forked xbatcher to let us filter patches. So you have to pip install from git.

In [ ]:
!pip install -q shap pytorch_lightning git+https://github.com/s-kganz/xbatcher.git@patch_filter_resample

In [ ]:
import torch
import xarray as xr
import xbatcher
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
import torch.nn.functional as F
import torchmetrics
from shap import DeepExplainer
import datetime
import warnings
from IPython.display import clear_output

from convlstm import ConvLSTM
from const import HOST_SPCODES

# If we have gpu available, use it
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()
torch.set_default_device(device)

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Prepare data

In [ ]:
ds = xr.open_zarr("../data_working/westmort.zarr/").compute()

# This model only requires the spatiotemporal context for mortality.
target_vars = list(filter(lambda x: x.endswith("_target"), ds.variables.keys()))
damage = ds[target_vars].to_dataarray(dim="variable").transpose("variable", "time", "y", "x")

# Create a mask from basal area
ba_vars = list(filter(lambda x: x.endswith("_ba"), ds.variables.keys()))
basal_area = ds[ba_vars].to_dataarray(dim="variable")

In [ ]:
# Free memory since we don't need the other covariates
del ds

We want to do a temporal train/valid split, but agent mortality is not distributed the same. We do a different split for each agent where at least 80% of pixels with mortality are in the training split.

In [ ]:
def get_split_idx(arr: xr.DataArray, q: float=0.8) -> int:
    ann_nonzero_px = (arr > 0).astype(np.float32).sum(dim=["x", "y"])
    tot_nonzero_px = ann_nonzero_px.sum()
    split_idx = np.where(np.cumsum(ann_nonzero_px) >= (tot_nonzero_px * q))[0][0] + 1
    return split_idx

Set up windowing parameters.

In [ ]:
# This is the full slice for each training instance.
# But the prediction label is the central 8x8 window
# in the last time step. This section must be fully
# Non-NA to be accepted.
WINDOW = dict(x=16, y=16, time=5)

# There is ~1 order of magnitude difference in data
# availability across agents. For generalist agents,
# use high overlap, otherwise use low overlap. This
# saves on windowing time and yields enough data.
GENERALIST_AGENTS = [
    "doug_fb_target",
    "fir_eng_target",
    "mtn_pb_target",
    "west_pb_target"
]
HIGH_OVERLAP = dict(x=10, y=10, time=4)
LOW_OVERLAP = dict(x=5, y=5, time=4)

# `INPUT` is the data cube the CNN sees, while `OUTPUT`
# is the small window at the last time step we are
# trying to predict.
INPUT_SELECTOR = dict(time=slice(None, -1))
TARGET_SELECTOR = dict(
    x=slice(WINDOW["x"]//2-1, (WINDOW["x"]//2)+1),
    y=slice(WINDOW["y"]//2-1, (WINDOW["y"]//2)+1),
    time=-1
)

# Epsilon to use for safe logit/expit transformations
LOGIT_EPS = 1e-3

# We need 64-bit floats for numerical precision with SHAP later
DTYPE = torch.float64

def patch_filter(ds: xr.DataArray, selector: dict) -> bool:
    patch = ds.isel(**selector)
    target = patch.isel(**TARGET_SELECTOR).values
    # Sum() on a bool dtype does not work correctly so cast to
    # a float.
    no_nulls = not np.any(np.isnan(target))
    has_damage = np.max(target) > 0
    
    return no_nulls and has_damage

def collator(patches: list[torch.Tensor], dtype=torch.float32) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.stack([
        torch.tensor(
            # Add a fake channel axis for compatability with the ConvLSTM
            # implementation we are using.
            np.expand_dims(patch.isel(**INPUT_SELECTOR).values, (1)),
            dtype=DTYPE
        )
        for patch in patches
    ])
    # Rescaling is not necessary because of batchnorm layer.
    X = torch.nan_to_num(X)

    y = torch.stack([
        torch.tensor(
            patch.isel(**TARGET_SELECTOR).values,
            dtype=DTYPE
        )
        for patch in patches
    ])
    # Scale to [0, 1]
    y = y / 100
    # Convert to logits to make SHAP more interpretable
    y = torch.special.logit(y, eps=LOGIT_EPS)

    return X, y

Functions to go from xarray object to torch Dataset.

In [ ]:
def make_bgen(
    damage: xr.DataArray, host_ba: xr.DataArray, 
    agent_var: str, ba_var: str
) -> xbatcher.BatchGenerator:
    # Only consider mortality from this agent in pixels where host trees
    # are present.
    damage_subset = damage.sel(variable=agent_var)
    damage_masked = damage_subset.where(host_ba.sel(variable=ba_var) > 0)

    # In specialist agents, masking can result in large areas where no
    # mortality is present that take forever to window. For speedup, identify
    # the bounding box where there is at least some mortality.
    nz_y, nz_x = np.nonzero(damage_masked.sum(dim=["time"]).values)
    nz_bbox = dict(
        x=slice(
            max(0, np.min(nz_x)-WINDOW["x"]), 
            np.max(nz_x)+WINDOW["x"]
        ),
        y=slice(
            max(0, np.min(nz_y)-WINDOW["y"]), 
            np.max(nz_y)+WINDOW["y"]
        )
    )
    damage_nz = damage_masked.isel(**nz_bbox)
    
    bgen = xbatcher.BatchGenerator(
        damage_nz,
        input_dims=WINDOW,
        input_overlap=LOW_OVERLAP if agent_var in GENERALIST_AGENTS else HIGH_OVERLAP,
        filter_fn=patch_filter
    )

    return bgen

Optional: Check how much data is available for each agent.

In [ ]:
class BatchGenDataset(Dataset):
    def __init__(self, bgen):
        self.bgen = bgen

    def __len__(self):
        return len(self.bgen)

    def __getitem__(self, idx):
        return self.bgen[idx]

Define model structure.

In [ ]:
class Model(pl.LightningModule):
    def __init__(self, input_spatial_size: int, input_dim: int=1, hidden_dim: int=16, **kwargs):
        super(Model, self).__init__()
        # Define model layers
        self.bn = torch.nn.BatchNorm3d(input_dim)
        self.convlstm = ConvLSTM(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            batch_first=True,
            **kwargs
        )
        self.conv1 = torch.nn.Conv2d(hidden_dim, hidden_dim//2, kernel_size=input_spatial_size//2+1)
        self.conv2 = torch.nn.Conv2d(hidden_dim//2, 1, kernel_size=input_spatial_size//2-1)
        self.linear = torch.nn.Linear(2, 2)

        # Define metrics
        self.train_nrmse = torchmetrics.regression.NormalizedRootMeanSquaredError()
        self.train_r     = torchmetrics.regression.PearsonCorrCoef()
        self.train_spear = torchmetrics.regression.SpearmanCorrCoef()

        self.valid_nrmse = torchmetrics.regression.NormalizedRootMeanSquaredError()
        self.valid_r     = torchmetrics.regression.PearsonCorrCoef()
        self.valid_spear = torchmetrics.regression.SpearmanCorrCoef()

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Torch expects the channel axis to be second, but convlstm expects it
        # to be third. So we have to permute the input, batchnorm it, and then
        # permute it back.
        # (N, T, C, H, W) -> (N, C, T, H, W)
        X = X.permute(0, 2, 1, 3, 4)
        X = self.bn(X)
        # (N, C, T, H, W) -> (N, T, C, H, W)
        X = X.permute(0, 2, 1, 3, 4)

        X = self.convlstm(X)[1][0][0]
        X = self.conv1(X)
        X = self.conv2(X)

        # Apply linear transfrom to better map onto logit space.
        X = self.linear(X)
        return X.squeeze()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.005)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.2, patience=3, min_lr=5e-5)
        return {"optimizer": optimizer, "lr_scheduler": scheduler, "monitor": "valid_loss"}

    def _get_loss(self, y, y_hat):
        loss = F.mse_loss(y, y_hat)
        return loss

    def _backtransform(self, y, eps=1e-3):
        '''
        Convert an output from logit space to 0-1.
        '''
        return torch.special.expit(y)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        y_hat = self.forward(X)

        y_tform = self._backtransform(y)
        y_hat_tform = self._backtransform(y_hat)

        loss = self._get_loss(y, y_hat)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)

        self.train_nrmse(y, y_hat)
        self.log("train_nrmse", self.train_nrmse, on_epoch=True, on_step=False)

        self.train_r(y_tform.view(-1), y_hat_tform.view(-1))
        self.log("train_r", self.train_r, on_epoch=True, on_step=False)

        self.train_spear(y_tform.view(-1), y_hat_tform.view(-1))
        self.log("train_spear", self.train_spear, on_epoch=True, on_step=False)
            
        return loss

    def validation_step(self, batch, batch_idx):
        X, y = batch
        y_hat = self.forward(X)

        y_tform = self._backtransform(y)
        y_hat_tform = self._backtransform(y_hat)

        loss = self._get_loss(y, y_hat)
        self.log("valid_loss", loss, prog_bar=True, on_epoch=True, on_step=False)

        self.valid_nrmse(y, y_hat)
        self.log("valid_nrmse", self.valid_nrmse, on_epoch=True, on_step=False)

        self.valid_r(y_tform.view(-1), y_hat_tform.view(-1))
        self.log("valid_r", self.valid_r, on_epoch=True, on_step=False)

        self.valid_spear(y_tform.view(-1), y_hat_tform.view(-1))
        self.log("valid_spear", self.valid_spear, on_epoch=True, on_step=False)

Run an experiment!

In [ ]:
MODEL_ARGS = dict(
    input_spatial_size=WINDOW["x"],
    num_layers=2,
    input_dim=1,
    hidden_dim=8,
    dropout=0.3,
    kernel_size=(3, 3),
    bias=True,
    return_all_layers=False
)

def run_experiment(agent):
    print("Starting", agent)
    
    ba_var = f"{agent}_ba"
    target_var = f"{agent}_target"

    print("Getting split year...")
    split_idx = get_split_idx(damage.sel(variable=target_var))
    print("Split year:", damage.time.isel(time=split_idx).dt.year.values)

    print("Windowing...")
    ds_train = damage.isel(time=slice(None, split_idx-WINDOW["time"]))
    ds_valid = damage.isel(time=slice(split_idx-WINDOW["time"], None))
    train_bgen = make_bgen(ds_train, basal_area, target_var, ba_var)
    valid_bgen = make_bgen(ds_valid, basal_area, target_var, ba_var)
    print(f"Found {len(train_bgen)} training samples.")
    print(f"Found {len(valid_bgen)} validation samples.")

    train_dataloader = DataLoader(
        BatchGenDataset(train_bgen),
        batch_size=16,
        collate_fn=collator,
        generator=torch.Generator(device=device),
        shuffle=True,
        drop_last=True
    )
    
    valid_dataloader = DataLoader(
        BatchGenDataset(valid_bgen),
        batch_size=16,
        collate_fn=collator,
        generator=torch.Generator(device=device),
        shuffle=False,
        drop_last=True
    )

    m = Model(**MODEL_ARGS).to(DTYPE)
    
    trainer = pl.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=30,
        callbacks=[
            pl.callbacks.EarlyStopping("valid_loss", min_delta=0.1, patience=5),
            pl.callbacks.ModelCheckpoint(),
            pl.callbacks.LearningRateMonitor(logging_interval="epoch")
        ],
        logger=pl.loggers.CSVLogger(save_dir="../data_working/", name="mort_convnet", version=agent + "_" + datetime.datetime.now().strftime("%Y-%m-%d-%H%M"))
    )

    trainer.fit(m, train_dataloader, valid_dataloader)
    m.logger.experiment.log_metrics(dict(train_n=len(train_bgen)))
    m.logger.experiment.log_metrics(dict(valid_n=len(valid_bgen)))

    return (agent, m, train_bgen, valid_bgen)

In [ ]:
warnings.filterwarnings("ignore", message="SpearmanCorrcoef")
agents = list(HOST_SPCODES.keys())
models_loaders = []

for agent in agents:
    models_loaders.append(run_experiment(agent))
    clear_output()

## SHAP analysis

In [ ]:
# SHAP is only possible on a scalar output, so we have to collapse the
# CNN output to a scalar.
class ShapAdapter(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module):
        super(ShapAdapter, self).__init__()
        self.base_model = base_model

    def forward(self, X):
        X = self.base_model(X)
        X = torch.mean(X, dim=[1, 2]).unsqueeze(-1)
        return X

In [ ]:
def run_shap(agent: str, model: pl.LightningModule, bgen: xbatcher.BatchGenerator):
    # Make a random sample of 100-200 background instances. This is more than
    # sufficient to get a good SHAP estimate according to their docs. But
    # we may have to discard samples that don't meet the additivity assumption
    # for SHAP.
    shap_samples = np.random.randint(low=0, high=len(bgen), size=min(200, len(bgen)))
    shap_X, _ = collator([bgen[int(i)] for i in shap_samples])
    shap_X = shap_X.to(device)
    
    # SHAP values are in logit space!!
    m_adapt = ShapAdapter(model).to(device)
    out = m_adapt(shap_X)
    de = DeepExplainer(m_adapt, shap_X)
    shap_values = de.shap_values(shap_X, check_additivity=False)
    
    # Shift expected values to have zero mean
    expected = out.detach().cpu().numpy()[:, 0]
    expected = expected - np.mean(expected)
    
    # SHAP values for the entire sample *should* sum to equal the model
    # output of that sample. Some are way off, probably because of numerical precision.
    sum_out = np.sum(shap_values.reshape(shap_values.shape[0], -1), 1)
    rel_delta = (sum_out - expected) / expected
    # plt.hist(rel_delta)
    
    # Only keep SHAPs within 1% of expected value.
    shap_values = shap_values[rel_delta<0.01, ...]
    shap_values = np.squeeze(shap_values)
    print(f"{agent} accepted {shap_values.shape[0]} samples")

    shap_da = xr.DataArray(
        shap_values,
        dims=("sample", "rel_time", "rel_y", "rel_x"),
        coords=dict(
            sample=np.arange(shap_values.shape[0]),
            rel_time=np.arange(shap_values.shape[1]) - WINDOW["time"] + 1,
            rel_x=np.arange(WINDOW["x"]) - (WINDOW["x"] // 2),
            rel_y=np.arange(WINDOW["y"]) - (WINDOW["y"] // 2)
        )
    ).expand_dims(agent=[agent])

    return shap_da

shap_arrs = [
    run_shap(agent, model, bgen)
    for (agent, model, _, bgen) in models_loaders
]  

In [ ]:
shap_combined = xr.combine_by_coords(shap_arrs, join="outer")
shap_combined.to_netcdf("../data_working/mort_convnet/mort_convnet_shap_by_agent.nc")

## Plots for fun

In [ ]:
shap_combined

In [ ]:
np.abs(shap_combined).mean(dim="sample").sum(dim="rel_time").plot(col="agent", col_wrap=3)

In [ ]:
mean_over_space = np.abs(shap_da).mean(dim=["sample", "time"])

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

fig, ax = plt.subplots()

window_y = shap_da.sizes["rel_y"]
window_x = shap_da.sizes["rel_x"]
cell_size_km = 3

x_ticks = np.arange(0, window_x, 2)
y_ticks = np.arange(0, window_y, 2)
x_offset = (x_ticks - (window_x//2)) * cell_size_km
y_offset = (y_ticks - (window_y//2)) * cell_size_km * -1

patch = matplotlib.patches.Rectangle((window_x//2-1.5, window_y//2-1.5), 2, 2, edgecolor="red", facecolor="none", linewidth=2, ls="--")
im = ax.imshow(mean_over_space.values)
ax.add_patch(patch)

ax.set_xticks(x_ticks-0.5, labels=x_offset)
ax.set_yticks(y_ticks-0.5, labels=y_offset)

fig.colorbar(im, label="Absolute value of SHAP")
plt.ylabel("Kilometers north of window center")
plt.xlabel("Kilometers east of window center")
plt.show()

In [ ]:
shap_values_by_time = np.abs(shap_da).mean(dim=["sample"])
print(shap_values_by_time.shape)

In [ ]:
plt.boxplot(shap_values_by_time.values.reshape(4, -1).T)
plt.xticks(ticks=[1, 2, 3, 4], labels=[4, 3, 2, 1])
plt.xlabel("Years prior to prediction")
plt.ylabel("Absolute value of SHAP")
plt.show()

In [ ]:
fig, (left, right) = plt.subplots(nrows=1, ncols=2, width_ratios=[1, 2], figsize=(8, 4))

left.boxplot(shap_values_by_time.values.reshape(4, -1).T)
left.set_xticks(ticks=[1, 2, 3, 4], labels=[4, 3, 2, 1])
left.set_xlabel("Years prior to prediction")
left.set_ylabel("Absolute value of SHAP")

patch = matplotlib.patches.Rectangle((window_x//2-1.5, window_y//2-1.5), 2, 2, edgecolor="red", facecolor="none", linewidth=2, ls="--")
im = right.imshow(mean_over_space.values)
right.add_patch(patch)

right.set_xticks(x_ticks-0.5, labels=x_offset)
right.set_yticks(y_ticks-0.5, labels=y_offset)

fig.colorbar(im, label="Absolute value of SHAP")
right.set_ylabel("Kilometers north of window center")
right.set_xlabel("Kilometers east of window center")

plt.tight_layout(w_pad=3.0)
plt.show()